# Parallel (orthogonal) regions

A parallel state has several regions that are all active simultaneously, each with its own current sub-state. This avoids the state explosion of a flat machine: three independent on/off flags need 3 x 2 = 6 states here instead of 2^3 = 8 product states (and the gap grows quickly).

`TextStyle` has `bold`, `italic` and `underline` regions.

```mermaid
stateDiagram-v2
    state editing {
        [*] --> bold_off
        bold_off --> bold_on: toggle_bold
        bold_on --> bold_off: toggle_bold
        --
        [*] --> italic_off
        italic_off --> italic_on: toggle_italic
        italic_on --> italic_off: toggle_italic
        --
        [*] --> underline_off
        underline_off --> underline_on: toggle_underline
        underline_on --> underline_off: toggle_underline
    }
```

In [1]:
from petrilab.fsm.parallel import TextStyle

REGIONS = ("bold", "italic", "underline")


def styles(sm):
    active = {s.id for s in sm.configuration}
    return tuple(f"{r}_on" in active for r in REGIONS)


sm = TextStyle()
styles(sm), sorted(s.id for s in sm.configuration)

((False, False, False),
 ['bold',
  'bold_off',
  'editing',
  'italic',
  'italic_off',
  'underline',
  'underline_off'])

Initially all three regions are active, each in its `_off` state, so every style is off.

## Independent toggles

In [2]:
sm = TextStyle()
steps = []
for event in ["toggle_bold", "toggle_underline", "toggle_bold"]:
    sm.send(event)
    steps.append((event, styles(sm)))
steps

[('toggle_bold', (True, False, False)),
 ('toggle_underline', (True, False, True)),
 ('toggle_bold', (False, False, True))]

Each event changes only its own region (tuples are bold, italic, underline); the other regions are untouched.

## All combinations are reachable

In [3]:
import itertools

seen = set()
for mask in itertools.product([False, True], repeat=3):
    sm = TextStyle()
    for region, on in zip(REGIONS, mask, strict=True):
        if on:
            sm.send(f"toggle_{region}")
    seen.add(styles(sm))
len(seen)

8

All 8 combinations are reached, confirming the regions are truly independent: the overall state is the product of the region states.